<a href="https://colab.research.google.com/github/lohithshankarappa/aiLab/blob/main/Week6PropositionalEntailment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import re

class Expr:
    """A generalized abstract syntax tree node for propositional formulas."""
    def __init__(self, op, *args):
        self.op = op
        self.args = args

    def __repr__(self):
        if not self.args:
            return str(self.op)
        if len(self.args) == 1:
            return f"{self.op}{self.args}"
        if len(self.args) == 2:
            return f"({self.args} {self.op} {self.args})"
        return str(self.op)

    def evaluate(self, model):
        """Recursively evaluates the expression based on the truth assignment model."""
        if not self.args:  # Variable / Symbol
            return model[self.op]

        args_eval = [arg.evaluate(model) for arg in self.args]
        if self.op == '~':
            return not args_eval[0]
        if self.op == '&':
            return args_eval[0] and args_eval[1]
        if self.op == '|':
            return args_eval[0] or args_eval[1]
        if self.op == '>>':
            return (not args_eval[0]) or args_eval[1]
        if self.op == '<<>>':
            return args_eval[0] == args_eval[1]
        raise ValueError(f"Unknown operator: {self.op}")

    def get_symbols(self):
        """Extracts all unique variable names from the expression tree."""
        if not self.args:
            return {self.op}
        symbols = set()
        for arg in self.args:
            symbols.update(arg.get_symbols())
        return symbols


def parse_expr(expr_str):
    """Parses a logical string formula into an Expr object tree."""
    s = expr_str.strip()
    s = s.replace('<<>>', '==').replace('>>', '<=')

    tokens = re.findall(r'[a-zA-Z_][a-zA-Z0-9_]*', s)
    if not tokens:
        raise ValueError(f"No propositional symbols found in: '{expr_str}'")

    namespace = {t: Expr(t) for t in tokens}

    class LogicWrapper:
        def __init__(self, obj): self.obj = obj
        def __invert__(self): return LogicWrapper(Expr('~', self.obj))
        def __or__(self, other): return LogicWrapper(Expr('|', self.obj, other.obj))
        def __and__(self, other): return LogicWrapper(Expr('&', self.obj, other.obj))
        def __le__(self, other): return LogicWrapper(Expr('>>', self.obj, other.obj))
        def __eq__(self, other): return LogicWrapper(Expr('<<>>', self.obj, other.obj))

    wrapped_ns = {k: LogicWrapper(v) for k, v in namespace.items()}
    s_wrapped = re.sub(r'([a-zA-Z_][a-zA-Z0-9_]*)', r'wrapped_ns["\1"]', s)

    try:
        return eval(s_wrapped, {"wrapped_ns": wrapped_ns}).obj
    except Exception:
        raise ValueError(f"Invalid syntax structure in expression: '{expr_str}'")


def PL_TRUE(sentence, model):
    """Returns True if a sentence holds within a model mapping."""
    if isinstance(sentence, list):
        return all(s.evaluate(model) for s in sentence)
    return sentence.evaluate(model)


# Global list to capture truth rows for printing the final table grid
truth_table_rows = []

def TT_ENTAILS(KB, alpha):
    """Truth-table enumeration algorithm for deciding propositional entailment."""
    global truth_table_rows
    truth_table_rows = []  # Clear previous runs

    KB_exprs = [parse_expr(s) if isinstance(s, str) else s for s in KB]
    alpha_expr = parse_expr(alpha) if isinstance(alpha, str) else alpha

    symbols_set = set()
    for s in KB_exprs:
        symbols_set.update(s.get_symbols())
    symbols_set.update(alpha_expr.get_symbols())

    symbols = sorted(list(symbols_set))

    # Run the recursive algorithm
    entails_result = TT_CHECK_ALL(KB_exprs, alpha_expr, symbols, {})

    # --- PRINT THE TRUTH TABLE GRID ---
    print("\n" + "="*80)
    print("                      GENERATED TRUTH TABLE (SLIDE ORDER)")
    print("="*80)

    # Print Table Header
    headers = symbols + ["KB Sentences Matrix...", "Combined KB", "Query (alpha)"]
    header_format = "".join([f"{{:<{max(len(h)+2, 8)}}}" for h in headers])
    print(header_format.format(*headers))
    print("-" * 80)

    # Print Each Row
    for row in truth_table_rows:
        row_vals = []
        # Symbols columns
        for sym in symbols:
            row_vals.append("false" if not row['model'][sym] else "true")

        # Individual KB sentences breakdown column text
        kb_status = ", ".join(["T" if s.evaluate(row['model']) else "F" for s in KB_exprs])
        row_vals.append(f"[{kb_status}]")

        # Combined KB total column
        kb_total = "true" if row['kb'] else "false"
        if row['kb']:
            kb_total += " <- Model"
        row_vals.append(kb_total)

        # Alpha Query column
        row_vals.append("true" if row['alpha'] else "false")

        print(header_format.format(*row_vals))

    print("="*80)
    return entails_result


def TT_CHECK_ALL(KB, alpha, symbols, model):
    """Recursive helper function evaluating complete binary truth configurations."""
    if not symbols:
        kb_val = PL_TRUE(KB, model)
        alpha_val = PL_TRUE(alpha, model)

        # Save current model valuation path to global table buffer
        truth_table_rows.append({'model': model.copy(), 'kb': kb_val, 'alpha': alpha_val})

        if kb_val:
            return alpha_val
        else:
            return True
    else:
        # FIX: Added [0] index to select the specific string element, not the entire slice list
        P = symbols[0]
        rest = symbols[1:]

        model_false = model.copy()
        model_false[P] = False

        model_true = model.copy()
        model_true[P] = True

        res_false = TT_CHECK_ALL(KB, alpha, rest, model_false)
        res_true = TT_CHECK_ALL(KB, alpha, rest, model_true)

        return res_false and res_true


# ==========================================
# INTERACTIVE USER INTERFACE CONSOLE
# ==========================================
if __name__ == "__main__":
    print("=" * 60)
    print("=" * 60)
    print("Allowed Operators: ~ (NOT), & (AND), | (OR), >> (Implication)")
    print("-" * 60)

    user_kb = []
    print("Enter your Knowledge Base (KB) sentences one by one.")
    print("Press [Enter] on an empty line when finished.")
    print("-" * 60)

    while True:
        sentence = input(f"Enter KB sentence #{len(user_kb) + 1}: ").strip()
        if not sentence:
            break
        user_kb.append(sentence)

    if not user_kb:
        print("\n Error: Knowledge base cannot be empty.")
        exit()

    print("-" * 60)
    user_query = input("Enter the query sentence (alpha): ").strip()

    try:
        entails = TT_ENTAILS(user_kb, user_query)
        print("\n" + "=" * 60)
        if entails:
            print("FINAL RESULT: KB ENTAILS alpha (KB ⊨ alpha) -> True")
        else:
            print("
            FINAL RESULT: KB DOES NOT ENTAIL alpha (KB ⊭ alpha) -> False")
        print("=" * 60)
    except Exception as e:
        print(f"\n Runtime Parsing Error: {e}")

Allowed Operators: ~ (NOT), & (AND), | (OR), >> (Implication)
------------------------------------------------------------
Enter your Knowledge Base (KB) sentences one by one.
Press [Enter] on an empty line when finished.
------------------------------------------------------------
Enter KB sentence #1: (A | C) & (B | ~C)
Enter KB sentence #2: 
------------------------------------------------------------
Enter the query sentence (alpha): A | B

                      GENERATED TRUTH TABLE (SLIDE ORDER)
A       B       C       KB Sentences Matrix...  Combined KB  Query (alpha)  
--------------------------------------------------------------------------------
false   false   false   [F]                     false        false          
false   false   true    [F]                     false        false          
false   true    false   [F]                     false        true           
false   true    true    [T]                     true <- Modeltrue           
true    false   false   [T]